# Mamba3 RSSM Smoke Test

This notebook tests the experimental `mamba3` RSSM path with synthetic tensors. It checks construction, posterior rollout, replay-style warmup, and open-loop imagination context flow.

## Colab / VSCode Notes

- In Colab, the notebook file can exist without the repo source files. The first code cell can clone your fork/branch into `/content/r2dreamer`.
- Before running in Colab, push this `r2dreamer` branch to your fork and set `REPO_URL` in the first code cell to your fork URL.
- If the source already exists somewhere else, set `R2DREAMER_DIR` in the first code cell to the folder that contains `rssm.py`.
- If dependencies are missing, install the repo first from the `r2dreamer` directory with `%pip install -e .`.
- The Mamba3 path requires a local/installable package exposing `mamba_ssm.modules.mamba3.Mamba3`.
- If you have a local `state-spaces/mamba` checkout in Colab, install it first with a command like `%pip install -e /content/mamba`, then restart the kernel.
- The core smoke tests do not require DeepMind Control Suite. The optional command at the end does.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab usually starts in /content with only the notebook present.
# If the r2dreamer source folder is not present, this cell can clone it.
# After pushing your branch, replace YOUR_USERNAME with your GitHub username.
REPO_URL = os.environ.get("R2DREAMER_REPO_URL", "https://github.com/YOUR_USERNAME/r2dreamer.git")
BRANCH = os.environ.get("R2DREAMER_BRANCH", "mamba3-rssm-experiment")
AUTO_CLONE = True

# If auto-detection fails, set this to the folder that contains rssm.py.
# Examples:
# R2DREAMER_DIR = "/content/r2dreamer"
# R2DREAMER_DIR = "/content/ssm-world/r2dreamer"
# R2DREAMER_DIR = "/content/drive/MyDrive/ssm-world/r2dreamer"
R2DREAMER_DIR = os.environ.get("R2DREAMER_DIR", "")
CLONE_DIR = Path(os.environ.get("R2DREAMER_CLONE_DIR", "/content/r2dreamer"))


def is_r2dreamer(path: Path) -> bool:
    return (path / "rssm.py").exists() and (path / "configs").is_dir()


def candidate_paths(start: Path):
    explicit = [Path(R2DREAMER_DIR).expanduser()] if R2DREAMER_DIR else []
    fixed = [
        Path("/content/r2dreamer"),
        Path("/content/ssm-world/r2dreamer"),
        Path("/content/drive/MyDrive/ssm-world/r2dreamer"),
        Path("/workspace/ssm-world/r2dreamer"),
    ]
    for base in [*explicit, start, *start.parents, *fixed]:
        yield base
        yield base / "r2dreamer"
        yield base / "ssm-world" / "r2dreamer"

    for root in [start, Path("/content"), Path("/workspace")]:
        if not root.exists():
            continue
        for pattern in ["r2dreamer", "*/r2dreamer", "*/*/r2dreamer", "*/*/*/r2dreamer"]:
            try:
                yield from root.glob(pattern)
            except OSError:
                pass


def find_r2dreamer(start: Path):
    seen = set()
    for candidate in candidate_paths(start):
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if is_r2dreamer(candidate):
            return candidate
    return None


def clone_r2dreamer():
    if "YOUR_USERNAME" in REPO_URL:
        raise FileNotFoundError(
            "The r2dreamer source folder is not available in this runtime. "
            "Set REPO_URL at the top of this cell to your fork, for example "
            "https://github.com/<your-user>/r2dreamer.git, or set R2DREAMER_DIR "
            "to an existing folder containing rssm.py and configs/."
        )
    if CLONE_DIR.exists() and is_r2dreamer(CLONE_DIR):
        return CLONE_DIR.resolve()
    if CLONE_DIR.exists() and any(CLONE_DIR.iterdir()):
        raise FileExistsError(
            f"Clone target {CLONE_DIR} already exists and is not an r2dreamer source folder. "
            "Set R2DREAMER_CLONE_DIR to an empty path or delete the stale folder."
        )
    cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        cmd += ["--branch", BRANCH]
    cmd += [REPO_URL, str(CLONE_DIR)]
    print("Cloning:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    return CLONE_DIR.resolve()


cwd = Path.cwd()
REPO_DIR = find_r2dreamer(cwd)
if REPO_DIR is None and AUTO_CLONE:
    REPO_DIR = clone_r2dreamer()
if REPO_DIR is None:
    raise FileNotFoundError(
        f"Could not find the r2dreamer source folder from cwd={cwd}. "
        "Clone/copy the repo first, or set R2DREAMER_DIR in this cell to the "
        "folder containing rssm.py and configs/."
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Using r2dreamer:", REPO_DIR)


In [ ]:
import importlib.util
import platform

print("Python:", platform.python_version())
if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is not installed in this runtime. In Colab, run `%pip install -e .` "
        "from the r2dreamer directory, then restart the kernel."
    )

import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Warning: Mamba3 kernels are usually CUDA-oriented. Use a GPU runtime if construction or forward pass fails.")

try:
    from mamba_ssm.modules.mamba3 import Mamba3  # noqa: F401
except Exception as exc:
    raise RuntimeError(
        "Mamba3 is not importable. The experimental RSSM core requires "
        "mamba_ssm.modules.mamba3.Mamba3. Install a Mamba package that includes "
        "Mamba3, then restart the kernel."
    ) from exc

print("Mamba3 import: ok")

In [ ]:
from types import SimpleNamespace as NS

import rssm


def make_rssm_config(device: str):
    return NS(
        core="mamba3",
        warmup=8,
        stoch=32,
        deter=512,
        hidden=128,
        discrete=16,
        img_layers=2,
        obs_layers=1,
        dyn_layers=1,
        blocks=8,
        act="SiLU",
        norm=True,
        unimix_ratio=0.01,
        initial="learned",
        device=device,
        mamba3=NS(
            context_len=16,
            n_layers=1,
            d_state=16,
            expand=1,
            headdim=128,
            is_mimo=False,
            mimo_rank=1,
            chunk_size=16,
            mlp_hidden_mult=1,
        ),
    )


device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = make_rssm_config(device)
model = rssm.RSSM(cfg, embed_size=128, act_dim=6).to(device)
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print("RSSM core:", cfg.core)
print("Device:", device)
print("Parameters:", f"{param_count:,}")
print("Uses context:", model.uses_context)

In [ ]:
torch.manual_seed(0)

B, T, E, A = 4, 8, 128, 6
embed = torch.randn(B, T, E, device=device)
action = torch.tanh(torch.randn(B, T, A, device=device))
reset = torch.zeros(B, T, dtype=torch.bool, device=device)
reset[:, 0] = True

stoch0, deter0 = model.initial(B)
context0 = model.initial_context(B)
initial = (stoch0, deter0, context0)

with torch.no_grad():
    stoch, deter, logit, context = model.observe(embed, action, initial, reset)

print("stoch:", tuple(stoch.shape))
print("deter:", tuple(deter.shape))
print("logit:", tuple(logit.shape))
print("context:", tuple(context.shape))

assert stoch.shape == (B, T, cfg.stoch, cfg.discrete)
assert deter.shape == (B, T, cfg.deter)
assert logit.shape == (B, T, cfg.stoch, cfg.discrete)
assert context.shape == (B, T, cfg.mamba3.context_len, cfg.deter)
assert torch.isfinite(stoch).all()
assert torch.isfinite(deter).all()
assert torch.isfinite(logit).all()
assert torch.isfinite(context).all()
print("Mamba3 observe smoke passed.")

In [ ]:
warmup = cfg.warmup
train_T = 6
total_T = warmup + train_T

embed2 = torch.randn(B, total_T, E, device=device)
action2 = torch.tanh(torch.randn(B, total_T, A, device=device))
reset2 = torch.zeros(B, total_T, dtype=torch.bool, device=device)
reset2[:, 0] = True

initial = (*model.initial(B), model.initial_context(B))

with torch.no_grad():
    warm_stoch, warm_deter, _, warm_context = model.observe(
        embed2[:, :warmup],
        action2[:, :warmup],
        initial,
        reset2[:, :warmup],
    )
    warmed_initial = (
        warm_stoch[:, -1].detach(),
        warm_deter[:, -1].detach(),
        warm_context[:, -1].detach(),
    )
    train_stoch, train_deter, train_logit, train_context = model.observe(
        embed2[:, warmup:],
        action2[:, warmup:],
        warmed_initial,
        reset2[:, warmup:],
    )

print("warm_context final:", tuple(warmed_initial[2].shape))
print("train_stoch:", tuple(train_stoch.shape))
print("train_deter:", tuple(train_deter.shape))
print("train_context:", tuple(train_context.shape))

assert warmed_initial[2].shape == (B, cfg.mamba3.context_len, cfg.deter)
assert train_stoch.shape == (B, train_T, cfg.stoch, cfg.discrete)
assert train_deter.shape == (B, train_T, cfg.deter)
assert train_context.shape == (B, train_T, cfg.mamba3.context_len, cfg.deter)
assert torch.isfinite(train_logit).all()
print("Mamba3 warmup smoke passed.")

In [ ]:
H = 5
future_action = torch.tanh(torch.randn(B, H, A, device=device))

with torch.no_grad():
    prior_stoch, prior_deter, prior_context = model.imagine_with_action(
        train_stoch[:, -1],
        train_deter[:, -1],
        future_action,
        train_context[:, -1],
    )

print("prior_stoch:", tuple(prior_stoch.shape))
print("prior_deter:", tuple(prior_deter.shape))
print("prior_context:", tuple(prior_context.shape))

assert prior_stoch.shape == (B, H, cfg.stoch, cfg.discrete)
assert prior_deter.shape == (B, H, cfg.deter)
assert prior_context.shape == (B, H, cfg.mamba3.context_len, cfg.deter)
assert torch.isfinite(prior_stoch).all()
assert torch.isfinite(prior_deter).all()
assert torch.isfinite(prior_context).all()
print("Mamba3 imagine smoke passed.")

In [ ]:
reset_now = torch.ones(B, dtype=torch.bool, device=device)

with torch.no_grad():
    reset_stoch, reset_deter, reset_logit, reset_context = model.obs_step(
        train_stoch[:, -1],
        train_deter[:, -1],
        action2[:, -1],
        embed2[:, -1],
        reset_now,
        train_context[:, -1],
    )

print("reset_stoch:", tuple(reset_stoch.shape))
print("reset_deter:", tuple(reset_deter.shape))
print("reset_context:", tuple(reset_context.shape))

assert reset_stoch.shape == (B, cfg.stoch, cfg.discrete)
assert reset_deter.shape == (B, cfg.deter)
assert reset_logit.shape == (B, cfg.stoch, cfg.discrete)
assert reset_context.shape == (B, cfg.mamba3.context_len, cfg.deter)
assert torch.isfinite(reset_context).all()
print("Mamba3 reset single-step smoke passed.")

## Optional End-to-End DMC Debug Run

This launches a short `dmc_walker_walk` run with the Mamba3 core. It requires `dm_control`, MuJoCo, Mamba3, and the full project dependencies.

In [ ]:
RUN_FULL_DMC = False

if RUN_FULL_DMC:
    import subprocess

    env = os.environ.copy()
    env.setdefault("MUJOCO_GL", "egl")
    train_device = "cuda:0" if torch.cuda.is_available() else "cpu"
    cmd = [
        sys.executable,
        "train.py",
        "env=dmc_proprio",
        "env.task=dmc_walker_walk",
        "env.env_num=1",
        "env.eval_episode_num=0",
        "env.steps=1000",
        "model=size_debug_mamba3",
        "model.rep_loss=r2dreamer",
        "model.compile=False",
        f"device={train_device}",
        "batch_size=4",
        "batch_length=16",
        "buffer.max_size=10000",
        "logdir=logdir/notebook_debug_mamba3",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, env=env)
else:
    print("Set RUN_FULL_DMC = True to launch the optional DMC debug run.")